In [ ]:
import medmnist
from medmnist import INFO, Evaluator
import torchvision.transforms as transforms

In [ ]:
data_flag = 'organamnist'
info = INFO[data_flag]
task = info['task']
n_channels = info['n_channels']
n_classes = len(info['label'])

DataClass = getattr(medmnist, info['python_class'])


# preprocessing
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

# load the data
train_dataset = DataClass(split='train', transform=data_transform, size=224)

In [ ]:
print(train_dataset)

In [ ]:
# montage
train_dataset.montage(length=4)

In [ ]:
import matplotlib.pyplot as plt

# find the class index for "bladder"
label_map = info['label']
bladder_idx = None
for k, v in label_map.items():
    if 'kidney-left' in str(v).lower():
        bladder_idx = int(k)
        break
if bladder_idx is None:
    raise ValueError(f'Bladder class not found. Available labels: {label_map}')

# collect some bladder images
num_to_show = 15
bladder_imgs = []
for img, target in train_dataset:
    # extract integer label from various possible target types
    try:
        t = int(getattr(target, 'item', lambda: target)())
    except Exception:
        try:
            t = int(target.squeeze())
        except Exception:
            t = int(target[0])
    if t == bladder_idx:
        bladder_imgs.append(img)
        if len(bladder_imgs) >= num_to_show:
            break

if not bladder_imgs:
    print('No bladder images found in the training split.')
else:
    cols = 4
    rows = (len(bladder_imgs) + cols - 1) // cols
    plt.figure(figsize=(3 * cols, 3 * rows))
    for i, img in enumerate(bladder_imgs):
        # unnormalize
        im = img * 0.5 + 0.5
        plt.subplot(rows, cols, i + 1)
        if im.ndim == 3 and im.shape[0] == 1:
            plt.imshow(im.squeeze(0).numpy(), cmap='gray')
        else:
            plt.imshow(im.permute(1, 2, 0).numpy(), cmap='gray')
        plt.title('kidney-left')
        plt.axis('off')
    plt.tight_layout()
    plt.show()